# 05 - Full Evaluation Benchmark

Benchmark AeroConform against standard anomaly detection baselines:
Isolation Forest, One-Class SVM, and threshold-based rules.

Metrics: FAR, Detection Rate, AUC-ROC, F1 score, and ROC curves.

## Setup

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

from aeroconform.config import AeroConformConfig

config = AeroConformConfig()
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## Generate Clean Training Trajectories

In [ ]:
rng = np.random.default_rng(42)


def generate_synthetic_trajectory(n_steps: int, rng: np.random.Generator) -> np.ndarray:
    """Generate a single realistic synthetic ADS-B trajectory."""
    lat = rng.uniform(config.bbox[0], config.bbox[1])
    lon = rng.uniform(config.bbox[2], config.bbox[3])
    alt = rng.uniform(15000, 40000)
    vel = rng.uniform(200, 500)
    hdg = rng.uniform(0, 360)
    vrate = 0.0
    traj = np.zeros((n_steps, 6))
    for t in range(n_steps):
        traj[t] = [lat, lon, alt, vel, hdg, vrate]
        lat += vel * np.cos(np.radians(hdg)) / 3600 / 60
        lon += vel * np.sin(np.radians(hdg)) / 3600 / 60 / np.cos(np.radians(lat))
        alt += vrate / 60
        vel = np.clip(vel + rng.normal(0, 1), 150, 600)
        hdg = (hdg + rng.normal(0, 0.3)) % 360
        vrate = rng.normal(0, 50)
    return traj


n_train = 30
n_test_clean = 10
clean_trajectories = [generate_synthetic_trajectory(200, rng) for _ in range(n_train + n_test_clean)]
train_trajectories = clean_trajectories[:n_train]
test_clean = clean_trajectories[n_train:]

print(f"Training trajectories: {len(train_trajectories)}")
print(f"Test clean trajectories: {len(test_clean)}")

## Generate Evaluation Set with Anomalies

Use the `AnomalyInjector` to create a mixed evaluation set with
5 anomaly types: GPS spoofing, position jumps, ghost aircraft,
replay attacks, and altitude manipulation.

In [ ]:
from aeroconform.data.synthetic_anomalies import generate_evaluation_set

test_trajs, test_labels, test_types = generate_evaluation_set(
    test_clean, anomalies_per_type=5, seed=42,
)

# Summarize
from collections import Counter
type_counts = Counter(test_types)
print("Evaluation set composition:")
for atype, count in sorted(type_counts.items()):
    print(f"  {atype}: {count}")
print(f"  Total: {len(test_trajs)}")

# Total anomalous timesteps
total_anomalous = sum(labels.sum() for labels in test_labels)
total_timesteps = sum(len(labels) for labels in test_labels)
print(f"\nAnomaly prevalence: {total_anomalous}/{total_timesteps} "
      f"({total_anomalous/total_timesteps:.1%})")

## Run Baselines

### Isolation Forest

In [ ]:
from aeroconform.evaluation.benchmark import IsolationForestBaseline
from aeroconform.evaluation.metrics import compute_all_metrics

iso_baseline = IsolationForestBaseline(contamination=0.05)
iso_baseline.fit(train_trajectories)

iso_preds = [iso_baseline.predict(t) for t in test_trajs]
iso_scores_list = [iso_baseline.score(t) for t in test_trajs]

iso_all_preds = np.concatenate(iso_preds)
iso_all_scores = np.concatenate(iso_scores_list)
iso_all_labels = np.concatenate(test_labels)

iso_metrics = compute_all_metrics(iso_all_preds, iso_all_scores, iso_all_labels)
print("Isolation Forest metrics:")
for k, v in iso_metrics.items():
    print(f"  {k}: {v:.4f}")

### One-Class SVM

In [ ]:
from aeroconform.evaluation.benchmark import OneClassSVMBaseline

svm_baseline = OneClassSVMBaseline(nu=0.05)
svm_baseline.fit(train_trajectories)

svm_preds = [svm_baseline.predict(t) for t in test_trajs]
svm_scores_list = [svm_baseline.score(t) for t in test_trajs]

svm_all_preds = np.concatenate(svm_preds)
svm_all_scores = np.concatenate(svm_scores_list)

svm_metrics = compute_all_metrics(svm_all_preds, svm_all_scores, iso_all_labels)
print("One-Class SVM metrics:")
for k, v in svm_metrics.items():
    print(f"  {k}: {v:.4f}")

### Threshold-Based Rules

In [ ]:
from aeroconform.evaluation.benchmark import ThresholdBaseline

thresh_baseline = ThresholdBaseline()

thresh_preds = [thresh_baseline.predict(t) for t in test_trajs]
thresh_scores_list = [thresh_baseline.score(t) for t in test_trajs]

thresh_all_preds = np.concatenate(thresh_preds)
thresh_all_scores = np.concatenate(thresh_scores_list)

thresh_metrics = compute_all_metrics(thresh_all_preds, thresh_all_scores, iso_all_labels)
print("Threshold baseline metrics:")
for k, v in thresh_metrics.items():
    print(f"  {k}: {v:.4f}")

### AeroConform Pipeline

In [ ]:
from aeroconform.models.trajectory_model import TrajectoryTransformer
from aeroconform.models.conformal import AdaptiveConformalDetector
from aeroconform.models.pipeline import AeroConformPipeline
from aeroconform.data.preprocessing import (
    delta_encode, compute_norm_stats, normalize, NormStats,
)

# Initialize model and detector
model = TrajectoryTransformer.from_config(config).to(device)
model.eval()
detector = AdaptiveConformalDetector.from_config(config)

# Compute norm stats from training data
train_deltas = [delta_encode(t) for t in train_trajectories if len(t) > 1]
norm_stats = compute_norm_stats(train_deltas)

# Calibrate on clean training data
cal_scores = []
for traj in train_trajectories:
    deltas = delta_encode(traj)
    normed = normalize(deltas, norm_stats)
    seq_len = config.seq_len
    if len(normed) >= seq_len:
        padded = normed[:seq_len]
    else:
        padded = np.zeros((seq_len, 6))
        padded[:len(normed)] = normed

    x = torch.from_numpy(padded).float().unsqueeze(0).to(device)
    with torch.no_grad():
        means, log_vars, log_weights, _ = model(x)

    for p in range(means.shape[1] - 1):
        ts = (p + 1) * config.patch_len
        te = ts + config.patch_len
        if te > len(deltas):
            break
        target = padded[ts:te].flatten()
        score = detector.compute_nonconformity_score(
            target, means[0, p].cpu().numpy(),
            log_vars[0, p].cpu().numpy(),
            log_weights[0, p].cpu().numpy(),
        )
        cal_scores.append(score)

detector.calibrate(np.array(cal_scores[:config.cal_window]))

# Build pipeline
pipeline = AeroConformPipeline(
    model=model, detector=detector, config=config,
    norm_stats=norm_stats, device=device,
)

# Evaluate
aero_all_preds = []
aero_all_scores = []

for traj, labels in zip(test_trajs, test_labels):
    result = pipeline.detect_anomalies(traj, update_calibration=False)
    n_timesteps = len(labels)

    # Map patch-level results back to timestep-level
    ts_preds = np.zeros(n_timesteps, dtype=int)
    ts_scores = np.zeros(n_timesteps)

    for r in result["results"]:
        p_idx = r["patch_idx"]
        start = (p_idx + 1) * config.patch_len
        end = min(start + config.patch_len, n_timesteps)
        if r["is_anomaly"]:
            ts_preds[start:end] = 1
        ts_scores[start:end] = r["score"]

    aero_all_preds.append(ts_preds)
    aero_all_scores.append(ts_scores)

aero_preds_concat = np.concatenate(aero_all_preds)
aero_scores_concat = np.concatenate(aero_all_scores)

aero_metrics = compute_all_metrics(aero_preds_concat, aero_scores_concat, iso_all_labels)
print("AeroConform metrics:")
for k, v in aero_metrics.items():
    print(f"  {k}: {v:.4f}")

## Metrics Comparison Table

In [ ]:
all_results = {
    "AeroConform": aero_metrics,
    "Isolation Forest": iso_metrics,
    "One-Class SVM": svm_metrics,
    "Threshold Rules": thresh_metrics,
}

metric_names = ["far", "detection_rate", "f1", "auc_roc"]
header = f"{'Method':<20}" + "".join(f"{m:>16}" for m in metric_names)
print(header)
print("=" * len(header))
for method, metrics in all_results.items():
    row = f"{method:<20}" + "".join(f"{metrics.get(m, 0.0):>16.4f}" for m in metric_names)
    print(row)

## ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

roc_data = {
    "AeroConform": (aero_scores_concat, iso_all_labels),
    "Isolation Forest": (iso_all_scores, iso_all_labels),
    "One-Class SVM": (svm_all_scores, iso_all_labels),
    "Threshold Rules": (thresh_all_scores, iso_all_labels),
}

colors = ["steelblue", "coral", "forestgreen", "darkorange"]

for (name, (scores, labels)), color in zip(roc_data.items(), colors):
    if len(np.unique(labels)) < 2:
        continue
    fpr, tpr, _ = roc_curve(labels, scores)
    auc = all_results[name]["auc_roc"]
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})", color=color, linewidth=2)

ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves: AeroConform vs Baselines")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Per-Anomaly-Type Detection Rates

In [ ]:
from aeroconform.evaluation.metrics import per_type_detection_rates

# AeroConform per-type rates
aero_type_rates = per_type_detection_rates(aero_all_preds, test_labels, test_types)

# Isolation Forest per-type rates
iso_type_rates = per_type_detection_rates(iso_preds, test_labels, test_types)

print("Per-anomaly-type detection rates:")
print(f"{'Type':<25} {'AeroConform':>12} {'IsoForest':>12}")
print("=" * 52)
all_types_set = sorted(set(aero_type_rates.keys()) | set(iso_type_rates.keys()))
for atype in all_types_set:
    aero_rate = aero_type_rates.get(atype, 0.0)
    iso_rate = iso_type_rates.get(atype, 0.0)
    print(f"{atype:<25} {aero_rate:>12.4f} {iso_rate:>12.4f}")

## Metric Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5))

methods = list(all_results.keys())
colors = ["steelblue", "coral", "forestgreen", "darkorange"]

for ax, metric in zip(axes, metric_names):
    values = [all_results[m].get(metric, 0.0) for m in methods]
    bars = ax.bar(range(len(methods)), values, color=colors, alpha=0.8)
    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels(methods, rotation=30, ha="right", fontsize=8)
    ax.set_title(metric.upper().replace("_", " "))
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3, axis="y")

    # Add value labels
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                f"{val:.3f}", ha="center", fontsize=8)

plt.suptitle("Anomaly Detection Metrics: All Methods", fontsize=14)
plt.tight_layout()
plt.show()

## Run Full Benchmark Suite

Use the `run_benchmark` function to run all baselines at once.

In [ ]:
from aeroconform.evaluation.benchmark import run_benchmark

benchmark_results = run_benchmark(
    train_trajectories=train_trajectories,
    test_trajectories=test_trajs,
    test_labels=test_labels,
    test_types=test_types,
)

print("Benchmark results (from run_benchmark):")
for method, metrics in benchmark_results.items():
    print(f"\n{method}:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

## Summary

This notebook demonstrated:

1. **Evaluation set generation** with 5 anomaly types via `AnomalyInjector`
2. **Baseline comparison**: Isolation Forest, One-Class SVM, threshold rules
3. **AeroConform pipeline** evaluation with conformal p-values
4. **Metrics**: FAR, detection rate, AUC-ROC, F1 score
5. **ROC curves** for all methods
6. **Per-type detection rates** for each anomaly category

Next: `06_live_demo.ipynb` demonstrates real-time anomaly detection.